In [1]:
from flask import Flask, render_template_string, request, redirect, flash
import sqlite3, threading, datetime

DB_NAME = "factory_management_new.db"

# ---------------- تاریخ شمسی امروز -----------------
def get_jalali_today():
    d = datetime.date.today()
    gy, gm, gd = d.year - 1600, d.month - 1, d.day - 1
    g_days = [0,31,59,90,120,151,181,212,243,273,304,334]
    g_day_no = 365*gy + (gy+3)//4 - (gy+99)//100 + (gy+399)//400
    g_day_no += g_days[gm] + gd
    if gm > 1 and ((gy%4==0 and gy%100!=0) or gy%400==0):
        g_day_no += 1
    j_day_no = g_day_no - 79
    j_np = j_day_no // 12053
    j_day_no %= 12053
    jy = 979 + 33*j_np + 4*(j_day_no//1461)
    j_day_no %= 1461
    if j_day_no >= 366:
        jy += (j_day_no-1)//365
        j_day_no = (j_day_no-1)%365
    for i, dm in enumerate([31]*6+[30]*5+[29]):
        if j_day_no < dm:
            return jy, i+1, j_day_no+1
        j_day_no -= dm

# ---------------- دیتابیس -----------------
def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS main_production(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT, actual INTEGER, plan INTEGER,
        commercial_count INTEGER, customer_delivery INTEGER)""")
    c.execute("""CREATE TABLE IF NOT EXISTS line_stoppages(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT, duration REAL, station TEXT,
        reason TEXT, responsible_unit TEXT,
        start_time TEXT, end_time TEXT)""")
    c.execute("""CREATE TABLE IF NOT EXISTS chassis_inventory(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT, factory_chassis INTEGER,
        customs_chassis INTEGER, kite_need INTEGER,
        side_need INTEGER)""")
    # جدول کسری قطعات
    c.execute("""CREATE TABLE IF NOT EXISTS parts_shortage(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date TEXT, item_code TEXT,
        item_desc TEXT, item_qty INTEGER)""")
    conn.commit()
    conn.close()

# ---------------- تب‌ها -----------------
SECTIONS = {
    "main_prod": {
        "title": "تولید و تحویل",
        "fields": [
            ("actual","تولید واقعی","number"),
            ("plan","برنامه تولید","number"),
            ("commercial_count","تولید تجاری شده","number"),
            ("customer_delivery","تحویل به مشتری","number")
        ]
    },
    "stoppages": {
        "title": "اطلاعات توقف خط",
        "fields": [
            ("duration","زمان توقف (ساعت)","text"),
            ("station","ایستگاه توقف","select"),
            ("reason","علت توقف","select"),
            ("responsible_unit","واحد مسئول","select"),
            ("start_time","ساعت شروع","time"),
            ("end_time","ساعت پایان","time")
        ]
    },
    "inventory": {
        "title": "موجودی شاسی‌ها",
        "fields": [
            ("factory_chassis","شاسی کارخانه","number"),
            ("customs_chassis","شاسی گمرک","number"),
            ("kite_need","نیاز کایت و بال","number"),
            ("side_need","نیاز ساید","number")
        ]
    },
    "parts_shortage": {
        "title": "کسری قطعات",
        "fields": [
            ("item_code","کد کالا","text"),
            ("item_desc","شرح کالا","text"),
            ("item_qty","تعداد","number"),
        ]
    }
}

STATIONS = ["بدون توقف","C01","C02","C04","C07","C12","K05","K07"]
REASONS  = ["برنامه مدیریتی","تست","تعمیرات","کمبود مواد","عملکرد اپراتور","سایر"]
UNITS    = ["تولید","مهندسی","کیفیت","مدیریت","تأمین خارج"]

# ---------------- HTML تم آبی کمرنگ -----------------
HTML = """
<!doctype html>
<html lang="fa" dir="rtl">
<head>
<meta charset="utf-8">
<title>ورود داده تولید</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/css/bootstrap.rtl.min.css" rel="stylesheet">
<style>
body{background:linear-gradient(to bottom,#e3f0fc,#fafdff 70%);font-family:'Vazirmatn',Tahoma,sans-serif;color:#2f6d9c}
.container{max-width:960px}
.card{background:#ffffffee;border-radius:18px;border:1.5px solid #b9defb;box-shadow:0 6px 24px rgba(60,160,240,.08)}
.nav-tabs{background:#d2edfc;border:none;border-radius:14px;padding:6px;box-shadow:0 4px 16px rgba(70,160,240,.08) inset}
.nav-tabs .nav-link{color:#398edb;font-weight:700;border:none;border-radius:10px 10px 0 0;margin:0 4px}
.nav-tabs .nav-link:hover{background:#e3f6ff;color:#2770af}
.nav-tabs .nav-link.active{background:#45a6f7;color:#fff;box-shadow:0 6px 18px rgba(69,166,247,.18)}
.tab-content{border:1px solid #b9defb;border-top:none;border-radius:0 0 14px 14px;padding:16px;background:#fafdff}
.form-control,.form-select{background:#fafdff!important;border:1px solid #b9defb!important;border-radius:10px;color:#2f6d9c}
.form-control:focus,.form-select:focus{border-color:#45a6f7!important;box-shadow:0 0 0 0.2rem rgba(69,166,247,.15)!important}
.bg-light{background:#e3f6ff!important;border:1px solid #b9defb!important;border-radius:12px;box-shadow:0 2px 10px rgba(180,220,250,.08)}
.btn-success{background:linear-gradient(to left,#45a6f7 92%,#8fd0ff 100%);border:none;color:#fff;font-weight:700;border-radius:12px}
</style>
</head>
<body>
<div class="container mt-4">
<div class="card p-4">
<h4 class="text-center mb-3">⚙️ سیستم ثبت داده‌های تولید</h4>
{% for m in get_flashed_messages() %}
<div class="alert py-2">{{m}}</div>
{% endfor %}
<ul class="nav nav-tabs mb-0">
{% for k,v in sections.items() %}
<li class="nav-item"><button class="nav-link {% if loop.first %}active{% endif %}" data-bs-toggle="tab" data-bs-target="#{{k}}">{{v.title}}</button></li>
{% endfor %}
</ul>
<div class="tab-content">
{% for k,v in sections.items() %}
<div class="tab-pane fade {% if loop.first %}show active{% endif %}" id="{{k}}">
<form method="post" action="/add/{{k}}">
<div class="row bg-light p-3 rounded mb-3">
  <div class="col-md-3">
    <label>سال</label>
    <select name="year" class="form-select">{% for y in range(1401,1407) %}<option {% if y==today[0] %}selected{% endif %}>{{y}}</option>{% endfor %}</select>
  </div>
  <div class="col-md-5">
    <label>ماه</label>
    <select name="month" class="form-select month" onchange="upd(this)">
      {% for i,m in [(1,'فروردین'),(2,'اردیبهشت'),(3,'خرداد'),(4,'تیر'),(5,'مرداد'),(6,'شهریور'),(7,'مهر'),(8,'آبان'),(9,'آذر'),(10,'دی'),(11,'بهمن'),(12,'اسفند')] %}
      <option value="{{'%02d'%i}}" {% if i==today[1] %}selected{% endif %}>{{m}}</option>{% endfor %}
    </select>
  </div>
  <div class="col-md-3">
    <label>روز</label>
    <select name="day" class="form-select day">{% for d in range(1,32) %}<option value="{{'%02d'%d}}" {% if d==today[2] %}selected{% endif %}>{{d}}</option>{% endfor %}</select>
  </div>
</div>
<div class="row">
{% for f,l,t in v.fields %}
  <div class="col-md-6 mb-3">
    <label>{{l}}</label>
    {% if t=="number" %}<input type="number" name="{{f}}" class="form-control">
    {% elif t=="time" %}<input type="time" name="{{f}}" class="form-control">
    {% else %}<input type="text" name="{{f}}" class="form-control">{% endif %}
  </div>
{% endfor %}
</div>
<div class="text-end"><button class="btn btn-success px-5">💾 ذخیره</button></div>
</form>
</div>
{% endfor %}
</div>
</div>
</div>
<script>
function upd(m){let d=m.closest('form').querySelector('.day');let mv=parseInt(m.value,10);let max=(mv<=6)?31:(mv<=11?30:29);let cur=d.value;d.innerHTML='';for(let i=1;i<=max;i++){let v=i<10?'0'+i:i;let o=new Option(i,v);if(v==cur)o.selected=true;d.add(o);}}
document.querySelectorAll('.month').forEach(upd);
</script>
<script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.2/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
"""

# ---------------- Flask -----------------
init_db()
app = Flask(__name__)
app.secret_key = "key"

@app.route("/")
def index():
    return render_template_string(HTML, sections=SECTIONS, today=get_jalali_today(), stations=STATIONS, reasons=REASONS, units=UNITS)

@app.route("/add/<t>", methods=["POST"])
def add(t):
    date = f"{request.form['year']}/{request.form['month']}/{request.form['day']}"
    data = {"date": date}
    for f,_,_ in SECTIONS[t]["fields"]:
        data[f] = request.form.get(f)
    tbl = {"main_prod":"main_production", "stoppages":"line_stoppages", "inventory":"chassis_inventory", "parts_shortage":"parts_shortage"}[t]
    cols = ",".join(data.keys())
    q = ",".join(["?"]*len(data))
    con = sqlite3.connect(DB_NAME)
    con.execute(f"INSERT INTO {tbl} ({cols}) VALUES ({q})", list(data.values()))
    con.commit(); con.close()
    flash("✅ اطلاعات با موفقیت ذخیره شد")
    return redirect("/")

threading.Thread(target=lambda: app.run(use_reloader=False), daemon=True).start()
print("✅ سیستم آماده است → http://127.0.0.1:5000")


✅ سیستم آماده است → http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [22/Dec/2025 21:44:38] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Dec/2025 21:44:39] "GET /favicon.ico HTTP/1.1" 404 -
